# ShelfWise LLM Training Bootstrap

This notebook prepares the first reproducible ShelfWise training artifacts: synthetic SFT rows, synthetic preference pairs, and an optional vLLM smoke check. It does not store API keys or train a large model locally.

## Goal

Use the repo-owned synthetic data generators to create a small, reviewable training dataset before starting expensive GPU fine-tuning. The remote ROCm/vLLM server can be tested from this notebook when an externally reachable base URL and API key are supplied through environment variables.

## Setup

Select the `Python (.venv ShelfWise AMD ACT II)` kernel. If you run this notebook on the remote Jupyter host instead of Windows, clone or upload this repo there first and install it with `python -m pip install -e .`.

Optional vLLM environment variables for a smoke check:

- `LLM_BASE_URL`: OpenAI-compatible vLLM URL, usually ending in `/v1`.
- `LLM_API_KEY`: bearer token generated on the vLLM host.
- `LLM_ROUTINE_MODEL`: `shelfwise-routine`.
- `LLM_STRONG_MODEL`: `shelfwise-strong`.

Do not paste secrets into a committed notebook cell.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"repo root: {ROOT}")
print(f"src import path present: {SRC.exists()}")

repo root: C:\Users\Admin\OneDrive\Documents\New folder\amd act II
src import path present: True


## Data

In [2]:
SEED = 42
SFT_ROWS = 64
PREFERENCE_ROWS = 32
OUTPUT_DIR = ROOT / "data" / "training"
SFT_PATH = OUTPUT_DIR / "shelfwise_sft_smoke.jsonl"
PREFERENCE_PATH = OUTPUT_DIR / "shelfwise_preference_smoke.jsonl"

print({
    "seed": SEED,
    "sft_rows": SFT_ROWS,
    "preference_rows": PREFERENCE_ROWS,
    "output_dir": str(OUTPUT_DIR),
})

{'seed': 42, 'sft_rows': 64, 'preference_rows': 32, 'output_dir': 'C:\\Users\\Admin\\OneDrive\\Documents\\New folder\\amd act II\\data\\training'}


In [3]:
from shelfwise_mlops import export_preference_jsonl, export_sft_jsonl
from shelfwise_synthdata import generate_agent_sft, generate_preference_pairs

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sft_count = export_sft_jsonl(generate_agent_sft(SEED, n=SFT_ROWS), SFT_PATH)
preference_count = export_preference_jsonl(
    generate_preference_pairs(SEED, n=PREFERENCE_ROWS),
    PREFERENCE_PATH,
)

summary = {
    "sft_count": sft_count,
    "preference_count": preference_count,
    "sft_path": str(SFT_PATH),
    "preference_path": str(PREFERENCE_PATH),
}
print(json.dumps(summary, indent=2))

{
  "sft_count": 64,
  "preference_count": 32,
  "sft_path": "C:\\Users\\Admin\\OneDrive\\Documents\\New folder\\amd act II\\data\\training\\shelfwise_sft_smoke.jsonl",
  "preference_path": "C:\\Users\\Admin\\OneDrive\\Documents\\New folder\\amd act II\\data\\training\\shelfwise_preference_smoke.jsonl"
}


In [4]:
def read_jsonl_preview(path: Path, limit: int = 2) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for index, line in enumerate(handle):
            if index >= limit:
                break
            rows.append(json.loads(line))
    return rows

preview = {
    "sft": read_jsonl_preview(SFT_PATH),
    "preference": read_jsonl_preview(PREFERENCE_PATH),
}
print(json.dumps(preview, indent=2))

{
  "sft": [
    {
      "messages": [
        {
          "content": "Assess expiry risk for synthetic SKU 2824.",
          "role": "user"
        }
      ],
      "output": {
        "agent": "expiry",
        "conclusion": "Review markdown with cited synthetic sources.",
        "confidence": 0.8,
        "sources": [
          "synthetic_case_0"
        ]
      },
      "synthetic": true
    },
    {
      "messages": [
        {
          "content": "Assess expiry risk for synthetic SKU 1409.",
          "role": "user"
        }
      ],
      "output": {
        "agent": "expiry",
        "conclusion": "Review markdown with cited synthetic sources.",
        "confidence": 0.8,
        "sources": [
          "synthetic_case_1"
        ]
      },
      "synthetic": true
    }
  ],
  "preference": [
    {
      "chosen": {
        "reason": "Recommendation lacks source coverage and needs review.",
        "verdict": "reject"
      },
      "messages": [
        {
          "content

## Optional vLLM Smoke Check

In [5]:
from shelfwise_inference import InferenceError, OpenAICompatibleInferenceClient, load_inference_config

config = load_inference_config()
print(json.dumps(config.to_public_dict(), indent=2))

if not config.base_url or not config.api_key_present:
    print("Skipping network smoke: set LLM_BASE_URL and LLM_API_KEY in the process environment.")
else:
    client = OpenAICompatibleInferenceClient(config)
    try:
        result = client.complete(
            agent="critic",
            system="You are the ShelfWise critic. Be concise and cite evidence gaps.",
            user="Review this recommendation: markdown SKU 4011 by 30 percent because expiry risk is high.",
            max_tokens=120,
        )
        public_result = result.to_dict()
        public_result["raw"] = "omitted"
        print(json.dumps(public_result, indent=2))
    except InferenceError as exc:
        print(f"Inference smoke failed: {exc}")

{
  "provider": "offline",
  "base_url_configured": false,
  "routine_model": "offline-routine",
  "strong_model": "offline-strong",
  "api_key_present": false,
  "routing": {
    "routine_agents": [
      "inventory",
      "expiry",
      "demand",
      "opportunity",
      "simulation"
    ],
    "strong_agents": [
      "critic",
      "executive",
      "orchestrator"
    ]
  }
}
Skipping network smoke: set LLM_BASE_URL and LLM_API_KEY in the process environment.


## Next Steps

1. Inspect the generated JSONL before treating it as training data.
2. Run the vLLM smoke check only after the remote endpoint is reachable from this notebook process.
3. For actual adapter training, use the ROCm host and a script-based training loop with checkpoints, not this notebook as the long-running trainer.
4. Keep preference/DPO data dormant until SFT quality is stable.